# Logistic Regression Classification

Logistic Regression không dự đoán giá tiền. Notebook này phân loại nhà thành nhóm dưới median và từ median `SalePrice`.

In [10]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_preprocessing import load_house_prices, prepare_features, build_preprocessor
from src.experiment_tracking import start_experiment, save_model, log_wandb

In [11]:
DATA_DIR = PROJECT_ROOT / 'data'
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments'
train, test = load_house_prices(DATA_DIR)
X, sale_price, X_test = prepare_features(train, test)
threshold = sale_price.median()
y = (sale_price >= threshold).astype('int8')
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
preprocessor, _, _ = build_preprocessor(X_train)

In [12]:
pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=3000, solver='liblinear', random_state=42))])
search = GridSearchCV(pipeline, {'classifier__C':[0.01,0.1,1,10], 'classifier__class_weight':[None,'balanced']}, scoring='roc_auc', cv=5, n_jobs=-1)
search.fit(X_train, y_train)
valid_probability = search.predict_proba(X_valid)[:, 1]
valid_prediction = (valid_probability >= 0.5).astype('int8')
metrics = {'roc_auc': float(roc_auc_score(y_valid, valid_probability)), 'accuracy': float(accuracy_score(y_valid, valid_prediction))}
print(search.best_params_, metrics)

{'classifier__C': 0.01, 'classifier__class_weight': 'balanced'} {'roc_auc': 0.9798742728466879, 'accuracy': 0.9212328767123288}


In [13]:
import os
os.environ['WANDB_MODE'] = 'online'

run_id, run_dir = start_experiment(EXPERIMENT_ROOT, 'logistic_regression', {'model':'LogisticRegression','task':'classification','threshold':float(threshold),'best_params':search.best_params_})
final_model = search.best_estimator_.fit(X, y)
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype('int8')
prediction_path = run_dir / 'predictions.csv'
pd.DataFrame({'Id': test['Id'], 'high_price_prediction': test_prediction, 'high_price_probability': test_probability}).to_csv(prediction_path, index=False)
metrics_path = run_dir / 'metrics.json'
from src.evaluation import save_metrics
save_metrics(metrics, metrics_path)
model_path = run_dir / 'model.pkl'
save_model(final_model, model_path)
mode = log_wandb(run_dir, 'house-price-classification', 'logistic_regression', {'threshold':float(threshold)}, metrics, [model_path, prediction_path, metrics_path, run_dir / 'config.json'])
print(f'Run: {run_dir}')
print(f'W&B mode: {mode}')

accuracy,▁
roc_auc,▁
accuracy,0.92123
roc_auc,0.97987


Run: c:\Users\ASUS\Data_Science_ Junior\DL\Tuan02\house_price\experiments\logistic_regression\20260919_180004
W&B mode: online
